# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

# 데이터 파악

## 원본 확인

1. 4개의 섹션으로 나뉜다.

    - 요약 페이지: 기본적인 연말정산 정보를 요약하여 제공.

    - 목차 페이지: 구체적인 정보에 대한 목차

    - 개정안 페이지: 개정된 정보만 따로 담아놓은 섹션

    - 상세 페이지: 상세한 정보 제공

2. 텍스트 데이터와 표 데이터로 나뉜다.

3. 기타 사항

   1. 병합된 셀이 많다.

   2. 특수문자(O) 같은 게 씹히는 경우가 있다.

   3. 타이틀을 일일이 달아줘야 할 것 같다.

   4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

   5. 이상한 글씨체 등 변칙적인 요소가 많다.


In [1]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()

try:
    from google.colab import drive, userdata
    # subprocess.run("pip", "install", "ipynbname")
    # import ipynbname
    # from IPython import get_ipython

    IS_COLAB_MODE = True
    print("코랩 모드")
    
    # ip = get_ipython()
    # if ip is not None:
    #     ip.run_line_magic('load_ext', 'colablinter')

    # ip.run_line_magic('clautofix', 'on')

    # path = ipynbname.path()
    
    # # %clreport [경로] 실행
    # ip.run_line_magic('clreport', path)

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"), override=True)
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


In [2]:
import re
import pandas as pd

# 표 데이터
import pdfplumber
from img2table.document import PDF
from img2table.ocr import TesseractOCR

# 텍스트 데이터
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 모델
from langchain_community.vectorstores import FAISS  # 벡터 저장소: 문서 임베딩 벡터를 저장하고 유사도 기반 검색을 위한 FAISS
from langchain_openai import OpenAIEmbeddings  # OpenAI의 텍스트 임베딩 모델(text-embedding-3 등)을 사용하는 모듈
from langchain_openai import ChatOpenAI  # OpenAI의 챗 모델(GPT-4, GPT-4o 등)을 사용하는 모듈
from langchain_core.prompts import ChatPromptTemplate  # LLM에 전달할 메시지를 템플릿 형태로 구성하기 위한 모듈
from langchain_core.output_parsers import StrOutputParser  # LLM의 응답 중 문자열만 깔끔히 추출해주는 파서
from transformers import TapexTokenizer, BartForConditionalGeneration

/Users/won/dev/00_codeit/0_mission/14_DL_RAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

for page in docs:
    page.metadata = {"page": page.metadata["page"]}

### 표 데이터

In [4]:
TABLES_BY_PAGE = PDF(PDF_PATH).extract_tables(
    ocr=TesseractOCR(n_threads=1, lang="eng"),
    min_confidence=40
)

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.54 : libtiff 4.7.1 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.5 zlib/1.2.12 liblzma/5.8.2 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.1 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.67.1


In [5]:
TABLES_BY_PAGE[4]

[ExtractedTable(title=상환기간 15년 이상
 상환기간 10년 이상, bbox=(670, 775, 1064, 870),shape=(2, 2))]

In [6]:
# 표 메타데이터 title 보정
for page_num, tables in TABLES_BY_PAGE.items():
    for table in tables:
        if table and table.title:
            if len(table.title) > 20:
                table.title = None

In [7]:
# 표 데이터 -> 텍스트 metadata에 merge
def trim_table(df: pd.DataFrame):
    df.columns = df.iloc[0]
    df = df[1:]
    df.reset_index(drop=True, inplace=True)

    return df


to_delete_list = list()

for page_num, tables in TABLES_BY_PAGE.items():
    if tables:
        docs[page_num].metadata["table"] = [
            {"table_num": f"{page_num}.{i}",
            "table": trim_table(table.df),
            "table_title": table.title} for i, table in enumerate(tables)
        ]

    else:
        docs[page_num].metadata["table"] = None
        to_delete_list.append(page_num)


for page_num in to_delete_list:
    del TABLES_BY_PAGE[page_num]

### 텍스트 데이터

In [8]:
# 문서 구조 기반 페이지 정리
DOCS_ABSTRACT = docs[:10]
DOCS_INDEX = docs[10:15]
DOCS_REVISED = docs[15:38]
DOCS_CONCRETE = docs[38:]

theme_map = {
    "abstract": {
        "page": range(10),
        "docs": DOCS_ABSTRACT
        },
    "index": {
        "page": range(10, 15),
        "docs": DOCS_INDEX
        },
    "revised": {
        "page": range(15, 38),
        "docs": DOCS_REVISED
        },
    "concrete": {
        "page": range(38, 426),
        "docs": DOCS_CONCRETE
        }
}

for theme in theme_map.keys():
    for doc in theme_map[theme]["docs"]:
        doc.metadata["theme"] = theme

### 요약 페이지(p.3-10)

In [9]:
# 메타데이터(chapter_title)
with pdfplumber.open(PDF_PATH) as pdf:
    for page in DOCS_ABSTRACT:
        title = ""

        page_num = page.metadata["page"] 

        head = pdf.pages[page_num].within_bbox((0, 0, 538, 130))
        tail = pdf.pages[page_num].within_bbox((0, 131, 538, 737))

        if head.extract_text():
            title = head.extract_text().replace("\n", " ")
        
        page.metadata["chapter_title"] = title    
        page.page_content = tail.extract_text()

In [10]:
print(DOCS_ABSTRACT[4])

page_content='(-) 연금보험료공제 공적연금(국민연금, 공무원연금, 군인연금, 사립학교교직원연금, 별정우체국연금, 국민연금과
직역연금의 연계에 관한 법률)의 근로자 부담금 : 전액 공제
102P
(-) 특별소득공제 보험료
104P 국민건강보험료·고용보험료·노인장기요양보험료 : 전액 공제
주택자금공제
주 택임차차입금 원리금상환액의 40% 공제 : 주택마련저축과 합하여 연 400만원 한도
장 기주택저당차입금 이자상환액 공제 : 연 600만원~2,000만원 한도
* 주택자금공제와 주택마련저축공제를 합하여 한도금액 계산(2012.1.1. 이후 차입분에 한해 적용)
상환기간 15년 이상 상환기간 10년 이상
고정금리 + 비거치식 고정금리 or 비거치식 기타 고정금리 or 비거치식
2,000만원 1,800만원 800만원 600만원
(-) 그 밖의 소득 개인연금저축 : 2000.12.31.까지 가입한 개인연금저축 납입액의 40% 공제(연 72만원 한도)
공제
소기업·소상공인 공제부금 : 소기업·소상공인공제에 가입하여 해당 연도에 납입한 금액
119P (근로소득금액 4천만원 이하 500만원, 1억원 이하 300만원, 1억원 초과 200만원 한도)
주택마련저축공제 : 청약저축·주택청약종합저축에 납입한 금액의 40% 공제(납입한도:
300만원)
중소기업창업투자조합 출자 등 소득공제 : 출자·투자분에 대해 투자금액의 10%(벤처
기업에 직접투자 3천만원이하 100%, 5천만원 이하 70%, 5천만원 초과 30%) 공제(종합
소득금액의 50% 한도로 하며, 벤처기업투자신탁에 대한 소득공제 금액은 300만원을
초과할 수 없음)
신용카드 등 사용금액 : 신용카드, 직불카드, 선불카드, 현금영수증 사용액의 합계액 중
총급여액의 25%를 초과하는 금액의 15%~40%를 소득공제
* 공 제한도 : 총급여 7천만원 이하는 300만원, 총급여 7천만원 초과는 250만원이며 공제한도 초과
금액이 있는 경우 아래 금액을 추가공제
 그 한도를 초과하는 금액과 전통시장 사용분에 공제율, 

### 목차 페이지(p.11-14)

In [11]:
index_dict = {
    "2024년귀속 연말정산 개정세법 요약": 1,
    "1부: 2024년 귀속 연말정산 중점 추진사항": {
        "Ⅰ 간소화서비스 전면 개편": {
            "page": 24,
            "sub": {
                "1. 주요 개선 내용": 24,
                "2. 소득금액 100만원(총급여 500만원) 산출 방법": 24,
                "3. 주의사항": 24,
            }
        },
        "Ⅱ 2024년 귀속 연말정산 주요 일정": {
            "page": 25,
            "sub": {
                "1. 회사의 연말정산 업무 일정": 26,
                "2. 원천징수의무자의 서류제출 의무": 30,
            }
        },
        "Ⅲ 원천징수의무자의 연말정산 중점 확인사항": {
            "page": 34,
            "sub": {
                "1. 근로소득 원천징수 중점 확인사항(연말정산 이전)": 34,
                "2. 소득·세액공제 증명서류 중점 확인사항(연말정산 시)": 36,
                "3. 연말정산 과다공제 주요 항목": 37,
                "4. 잘못된 소득·세액공제에 따른 가산세 부담": 44,
            }
        },
    },
    "2부: 근로소득 연말정산": {
        "Ⅰ 근로소득": {
            "page": 48,
            "sub": {
                "1. 근로소득의 범위(소법 §20)": 48,
                "2. 비과세 근로소득 등": 52,
                "3. 일용근로소득과 일반근로소득의 구분": 73,
                "4. 근로소득의 수입시기(소령 §49)": 75,
                "5. 근로소득 수입금액 계산": 76,
            }
        },
        "Ⅱ 근로소득 원천징수 및 연말정산": {
            "page": 78,
            "sub": {
                "1. 근로소득 원천징수 의무": 78,
                "2. 근로소득 원천징수 및 연말정산 절차": 80,
                "3. 특수한 경우의 연말정산": 82,
                "4. 연말정산 시기": 86,
                "5. 비거주자의 연말정산": 87,
                "6. 외국인의 연말정산(조특법 §18의2)": 90,
            }
        },
        "Ⅲ 근로소득공제, 인적공제, 연금보험료공제": {
            "page": 93,
            "sub": {
                "1. 근로소득공제(소법 §47)": 93,
                "2. 인적공제": 94,
                "3. 연금보험료공제(소법 §51의3)": 102,
            }
        },
        "Ⅳ 특별소득공제(소법 §52)": {
            "page": 104,
            "sub": {
                "1. 특별소득공제 개요": 105,
                "2. 보험료공제(소법 §52 ①)": 106,
                "3. 주택자금공제(소법 §52 ④, ⑤)": 106,
            }
        },
        "Ⅴ 그 밖의 소득공제(조특법)": {
            "page": 119,
            "sub": {
                "1. 개인연금저축 소득공제": 119,
                "2. 소기업·소상공인 공제부금 소득공제(조특법 §86의3)": 120,
                "3. 주택마련저축 납입액 소득공제(조특법 §87 ②)": 121,
                "4. 벤처투자조합 출자 등에 대한 소득공제(조특법 §16)": 124,
                "5. 신용카드 등 사용금액 소득공제(조특법 §126의2)": 130,
                "6. 우리사주조합 출연금 소득공제(조특법 §88의4)": 137,
                "7. 고용유지중소기업 근로자 소득공제(조특법 §30의3)": 138,
                "8. 장기집합투자증권저축 소득공제(조특법 §91의16)": 141,
                "9. 청년형 장기집합투자증권저축 소득공제(조특법 §91의20)": 143,
                "10. 소득세 소득공제 종합한도(조특법 §132의2)": 145,
            }
        },
        "Ⅵ 세액감면(공제) 및 농어촌특별세": {
            "page": 146,
            "sub": {
                "1. 소득세법에 따른 세액감면(소법 §59의5)": 146,
                "2. 조세조약에 따른 세액감면": 146,
                "3. 조세특례제한법에 따른 세액감면": 148,
                "4. 근로소득세액공제": 159,
                "5. 결혼세액공제": 160,
                "6. 자녀세액공제": 161,
                "7. 연금계좌세액공제": 162,
                "8. 특별세액공제(보험료, 의료비, 교육비, 기부금)": 166,
                "9. 월세액 세액공제": 200,
                "10. 납세조합 공제": 202,
                "11. 주택자금차입금 이자세액공제": 202,
                "12. 외국납부세액공제": 204,
                "13. 농어촌특별세": 207,
            }
        },
    },
    "3부: 연말정산 종합사례 및 서식 작성방법": {
        "Ⅰ 2024년 귀속 연말정산 종합사례": {
            "page": 210,
            "sub": {
                "1. 소득·세액공제금액 계산": 210,
                "2. 소득·세액공제신고서 작성": 218,
                "3. 의료비지급명세서 작성": 225,
                "4. 기부금명세서 작성": 227,
                "5. 신용카드 등 소득공제 신청서 작성": 230,
            }
        },
        "Ⅱ 근로소득 원천징수영수증(지급명세서) 작성요령": 231,
        "Ⅲ 원천징수이행상황신고서 작성사례": 247,
        "Ⅳ 수정 원천징수이행상황신고서 작성사례(과다공제)": 253,
        "Ⅴ 근로소득 경정청구 사례": 275,
        "Ⅵ 홈택스를 이용한 연말정산 신고(근로소득 지급명세서 제출)": {
            "page": 279,
            "sub": {
                "1. 지급명세서 전자제출": 279,
                "2. 지급명세서 전자제출 Q&A": 300,
                "3. 원천징수이행상황 신고 및 환급 신청": 307,
            }
        },
        "Ⅶ ICL홈페이지를 이용한 상환금명세서 전자신고": 320,
    },
    "4부: 사업소득 연금소득 연말정산": {
        "Ⅰ 사업소득 연말정산": 334,
        "Ⅱ 연금소득 연말정산": 341,
    },
    "5부: 종교인 소득 연말정산": {
        "Ⅰ 종교인소득이란?": 354,
        "Ⅱ 종교관련종사자": 357,
        "Ⅲ 종교단체": 358,
        "Ⅳ 종교인소득(기타소득)에 대한 연말정산": 359,
    },
    "부록": {
        "1. 연말정산 관련 서비스": 368,
        "2. 연말정산간소화 서비스": 385,
        "3. 간소화자료 일괄제공 서비스": 394,
        "4. 맞벌이 부부 연말정산": 399,
        "5. 연말정산 주요 용어 설명": 400,
        "6. 소득·세액공제신고서 첨부서류": 404,
    },
}

In [12]:
LEVEL_NAMES = ["부", "대단원", "중단원", "소단원"]

def dig_toc(d, depth=0):
    """toc 딕셔너리를 재귀 탐색하여 (제목, 페이지, 계층레벨, 계층명) 리스트를 반환"""
    results = []
    for key, value in d.items():
        level = LEVEL_NAMES[depth] if depth < len(LEVEL_NAMES) else f"Level {depth}"

        if isinstance(value, int):
            results.append({"title": key, "page": value, "depth": depth, "level": level})

        elif isinstance(value, dict) and "page" in value:
            results.append({"title": key, "page": value["page"], "depth": depth, "level": level})
            if "sub" in value:
                results.extend(dig_toc(value["sub"], depth + 1))

        elif isinstance(value, dict):
            results.append({"title": key, "page": None, "depth": depth, "level": level})
            results.extend(dig_toc(value, depth + 1))

    return results

toc_flat = dig_toc(index_dict)

In [22]:
root_title, trunk_title, leaf_title = "", "", ""

index_map = {i: [] for i in range(len(DOCS_REVISED) + len(DOCS_CONCRETE))}

for dict in toc_flat:
    depth = dict["depth"]

    if depth == 0:
        root_title = dict["title"]

    elif depth == 1:
        trunk_title = dict["title"]

    else:
        leaf_title = dict["title"]

    if dict["page"]:
        c_num = dict["page"]

    index_map[c_num].append([root_title, trunk_title, leaf_title])

current = ["", "", ""]
for i in range(len(index_map)):
    if index_map[i]:
        current = index_map[i][-1]
    else:
        index_map[i].append(current)


tmp = index_map[1][0]
for i in range(1, 24):
    index_map[i] = tmp

In [23]:
index_map

{0: [['', '', '']],
 1: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 2: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 3: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 4: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 5: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 6: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 7: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 8: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 9: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 10: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 11: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 12: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 13: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 14: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 15: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 16: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 17: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 18: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 19: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 20: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 21: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 22: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 23: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 24: [['1부: 2024년 귀속 연말정산 중점 추진사항', 'Ⅰ 간소화서비스 전면 개편', ''],
  ['1부: 2024년 귀속 연말정산 중점 추진사항', '

In [24]:
for key, chapter_list in index_map.items():
    if type(chapter_list[0]) == list and len(chapter_list) > 1:
        result = []

        for values in zip(*chapter_list):
            s = set(values)
            result.append(s.pop() if len(s) == 1 else s)

        index_map[key] = result


    elif type(chapter_list[0]) == list and len(chapter_list) == 1:
        index_map[key] = chapter_list[0]

In [25]:
index_map

{0: ['', '', ''],
 1: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 2: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 3: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 4: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 5: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 6: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 7: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 8: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 9: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 10: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 11: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 12: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 13: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 14: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 15: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 16: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 17: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 18: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 19: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 20: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 21: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 22: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 23: ['2024년귀속 연말정산 개정세법 요약', '', ''],
 24: ['1부: 2024년 귀속 연말정산 중점 추진사항',
  'Ⅰ 간소화서비스 전면 개편',
  {'', '1. 주요 개선 내용', '2. 소득금액 100만원(총급

In [26]:
for i in range(len(index_map)):
    
    space = len(DOCS_REVISED)

    if i < space:
        DOCS_REVISED[i].metadata["root_title"] = index_map[i][0]
        DOCS_REVISED[i].metadata["trunk_title"] = index_map[i][1]
        DOCS_REVISED[i].metadata["leaf_title"] = index_map[i][2]

    else:
        idx = i - space
        DOCS_CONCRETE[idx].metadata["root_title"] = index_map[i][0]
        DOCS_CONCRETE[idx].metadata["trunk_title"] = index_map[i][1]
        DOCS_CONCRETE[idx].metadata["leaf_title"] = index_map[i][2]

In [27]:
for i, doc in enumerate(DOCS_CONCRETE):
    print(i, doc.metadata["leaf_title"])

0 
1 {'', '1. 주요 개선 내용', '3. 주의사항', '2. 소득금액 100만원(총급여 500만원) 산출 방법'}
2 3. 주의사항
3 1. 회사의 연말정산 업무 일정
4 1. 회사의 연말정산 업무 일정
5 1. 회사의 연말정산 업무 일정
6 1. 회사의 연말정산 업무 일정
7 2. 원천징수의무자의 서류제출 의무
8 2. 원천징수의무자의 서류제출 의무
9 2. 원천징수의무자의 서류제출 의무
10 2. 원천징수의무자의 서류제출 의무
11 {'1. 근로소득 원천징수 중점 확인사항(연말정산 이전)', '2. 원천징수의무자의 서류제출 의무'}
12 1. 근로소득 원천징수 중점 확인사항(연말정산 이전)
13 2. 소득·세액공제 증명서류 중점 확인사항(연말정산 시)
14 3. 연말정산 과다공제 주요 항목
15 3. 연말정산 과다공제 주요 항목
16 3. 연말정산 과다공제 주요 항목
17 3. 연말정산 과다공제 주요 항목
18 3. 연말정산 과다공제 주요 항목
19 3. 연말정산 과다공제 주요 항목
20 3. 연말정산 과다공제 주요 항목
21 4. 잘못된 소득·세액공제에 따른 가산세 부담
22 4. 잘못된 소득·세액공제에 따른 가산세 부담
23 4. 잘못된 소득·세액공제에 따른 가산세 부담
24 4. 잘못된 소득·세액공제에 따른 가산세 부담
25 {'4. 잘못된 소득·세액공제에 따른 가산세 부담', '1. 근로소득의 범위(소법 §20)'}
26 1. 근로소득의 범위(소법 §20)
27 1. 근로소득의 범위(소법 §20)
28 1. 근로소득의 범위(소법 §20)
29 2. 비과세 근로소득 등
30 2. 비과세 근로소득 등
31 2. 비과세 근로소득 등
32 2. 비과세 근로소득 등
33 2. 비과세 근로소득 등
34 2. 비과세 근로소득 등
35 2. 비과세 근로소득 등
36 2. 비과세 근로소득 등
37 2. 비과세 근로소득 등
38 2. 비과세 근로소득 등
39 2. 비과세 근로소득 등
40 2. 비과세 근로소득 등
41 2. 비과세 근로소득 등
42 2. 비

### 개정안 페이지(p.15-38)

In [ ]:
# split 구분: 개정안 항목
chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separators=[r"\d+\s+.*?\n\(.*"],
    is_separator_regex=True,
    keep_separator=True
)

DOCS_REVISED = chapter_splitter.split_documents(DOCS_REVISED)

# 메타데이터(chapter_title)
del_list = list()
for chapter in DOCS_REVISED:
    if chapter.page_content == "원천징수의무자를 위한 \n2024년 연말정산 신고안내":
        del_list.append(chapter)
    elif chapter.page_content == "01. 2024년 귀속 연말정산 개정세법 요약":
        del_list.append(chapter)

    else:
        match = re.search(r"^\d+\s+(.*?)\n\(.*", chapter.page_content, re.DOTALL)
        if match:
            chapter_title = match.group(1).strip()
            chapter.metadata["chapter_title"] = chapter_title
            chapter.page_content = chapter.page_content.replace(chapter_title, "")

for del_ in del_list:
    DOCS_REVISED.remove(del_)

In [ ]:
print(DOCS_REVISED[1])

page_content='1  
(소득세법 제12조 제3호 마목, 같은 법 시행령 제10조의2)
<개정취지> 육아휴직 지원
종          전 개          정
▢ 근로소득에서 비과세되는 육아휴직 급여·수당 ▢ 비과세 소득 확대
○ ｢고용보험법｣에 따라 받는 육아휴직급여 ○ (좌  동)
○ 공무원 또는 ｢사립학교교직원 연금법｣, ｢별정우체국법｣을
적용받는 사람이 관련 법령에 따라 받는 육아휴직수당
<추  가>    - 사립학교 직원이 사립학교 정관 등에 의해 지급받는 
월 150만원 이하의 육아휴직수당
<적용시기> 2024.1.1. 이후 지급받는 분부터 적용
' metadata={'page': 16, 'table': None, 'chapter_title': '육아휴직수당 비과세 적용대상 확대 및 범위 규정'}


### 상세 페이지(p.39-426)

In [ ]:
concrete_map = {
    "2024년 귀속 연말정산 중점 추진사항": range(37, 62),
    "근로소득 연말정산": range(62, 224),
    "연말정산 종합사례 및 서식 작성방법": range(224, 348),
    "사업소득·연금소득 연말정산": range(348, 368),
    "종교인 소득 연말정산": range(368, 382),
    "연말정산 관련 서비스": range(382, 400),
    "연말정산 간소화 서비스": range(400, 409),
    "간소화자료 일괄제공 서비스": range(409, 414),
    "맞벌이부부 연말정산": range(414, 415),
    "연말정산 주요 용어 설명": range(415, 419),
    "소득·세액공제신고서 첨부서류": range(419, 426),
}

tmp_num_map = dict()
for key, value in concrete_map.items():
    for num in value:
        tmp_num_map[num] = key


# 메타데이터(chapter_title)
for page in DOCS_CONCRETE:
    page.metadata["chapter_title"] = tmp_num_map[page.metadata["page"]]

# 실험 계획

자료가 목차와 카테고리를 기반으로 탐색하면 유리한 구조로 되어 있으므로,

메타데이터를 활용하여 카테고리 정보를 가진 데이터와 그렇지 않은 데이터의 성능을 비교한다.

## RAG 설계

0. 문장 데이터와 표 데이터로 나눈다.
1. 문장 데이터는 OPENAI 모델이 진행.
2. 표 데이터는 https://huggingface.co/microsoft/tapex-base-finetuned-wikisql
3. 허위 정보를 알려줘서는 안 되니, 모르는 건 모른다고 하자.
4. 참고한 페이지 정보를 같이 출력해주어 유저가 더블 체크할 수 있도록 하자.

pdf 육안 확인 + df로 바꿔보니 여간 복잡한 게 아니다.

1. 병합된 셀이 많다.
2. 특수문자(O) 같은 게 씹히는 경우가 있다.
3. 타이틀을 일일이 달아줘야 할 것 같다.
4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

In [ ]:
def put_id(docs):
    for idx, doc in enumerate(docs):
        doc.metadata["id"] = idx

put_id(DOCS_ABSTRACT), put_id(DOCS_INDEX), put_id(DOCS_REVISED), put_id(DOCS_CONCRETE)

(None, None, None, None)

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")


vector_dir = os.path.join(DATA_DIR, "vector")
os.makedirs(vector_dir, exist_ok=True)

try:
    vector_abstract = FAISS.load_local(folder_path=vector_dir, embeddings=embedding_model, index_name="abstract", allow_dangerous_deserialization=True)
    vector_concrete = FAISS.load_local(folder_path=vector_dir, embeddings=embedding_model, index_name="concrete", allow_dangerous_deserialization=True)
    vector_revised = FAISS.load_local(folder_path=vector_dir, embeddings=embedding_model, index_name="revised", allow_dangerous_deserialization=True)
    vector_index = FAISS.load_local(folder_path=vector_dir, embeddings=embedding_model, index_name="index", allow_dangerous_deserialization=True)

except:
    vector_abstract = FAISS.from_documents(DOCS_ABSTRACT, embedding_model)
    vector_concrete = FAISS.from_documents(DOCS_CONCRETE, embedding_model)
    vector_revised = FAISS.from_documents(DOCS_REVISED, embedding_model)
    vector_index = FAISS.from_documents(DOCS_INDEX, embedding_model)

    vector_abstract.save_local(folder_path=vector_dir, index_name="abstract")
    vector_concrete.save_local(folder_path=vector_dir, index_name="concrete")
    vector_revised.save_local(folder_path=vector_dir, index_name="revised")
    vector_index.save_local(folder_path=vector_dir, index_name="index")

retriever_abstract = vector_abstract.as_retriever(search_kwargs={"k": 4})
retriever_concrete = vector_concrete.as_retriever(search_kwargs={"k": 4})
retriever_revised = vector_revised.as_retriever(search_kwargs={"k": 4})
retriever_index = vector_index.as_retriever(search_kwargs={"k": 4})

######
text_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

table_model_dir = os.path.join(DATA_DIR, "model", "tapex_model")
table_model = BartForConditionalGeneration.from_pretrained(table_model_dir)
table_tokenizer = TapexTokenizer.from_pretrained(table_model_dir)

parser = StrOutputParser()

In [ ]:
question = [
    "기부금 공제 때 주의할 점은?",
    "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?",
    "연말 정산 때 비거주자가 주의할 점을 알려 줘."
]